In [7]:
from kfp.dsl import component
from kfp import compiler, dsl
from kfp.dsl import Output, Dataset, Input
import os
from dotenv import load_dotenv
from google.cloud import aiplatform
load_dotenv(override=True)

True

In [3]:
PROJECT_ID = os.environ["PROJECT_ID"]
PIPELINE_ROOT = os.environ["PIPELINE_ROOT"]

In [99]:
IMAGE_URI = f"us-central1-docker.pkg.dev/{PROJECT_ID}/rf-pytorch/rf-arcface:latest"

In [87]:
@dsl.component(
    base_image="python:3.12.9",
    packages_to_install=["gdown", "google-cloud-storage"],
)
def download_data(dataset: Output[Dataset]):
    from pathlib import Path
    import zipfile
    import gdown
    from google.cloud import storage

    gdrive_url = "https://drive.google.com/uc?id=1KxNCrXzln0lal3N4JiYl9cFOIhT78y1l"
    
    bucket_name = "rf-pytorch-bucket"
    blob_name = "datasets/faces_webface_112x112/dataset.zip"

    zip_path = Path("/tmp/dataset.zip")
    output_dir = Path(dataset.path)
    output_dir.mkdir(parents=True, exist_ok=True)

    success = False

    try:
        print("Attempting to download dataset from Google Drive...")
        gdown.download(gdrive_url, str(zip_path), quiet=False)
        if zipfile.is_zipfile(zip_path):
            success = True
            print("Successfully downloaded from Google Drive.")
        else:
            print("Downloaded file from Google Drive is not a valid ZIP.")
    except Exception as e:
        print(f"Google Drive download failed: {e}")

    if not success:
        print(f"Falling back to GCS (gs://{bucket_name}/{blob_name})...")
        try:
            client = storage.Client()
            bucket = client.bucket(bucket_name)
            blob = bucket.blob(blob_name)
            blob.download_to_filename(str(zip_path))
            
            if not zipfile.is_zipfile(zip_path):
                raise ValueError("Downloaded file from GCS is not a valid ZIP archive.")
            print("Successfully downloaded from GCS.")
        except Exception as e:
            raise RuntimeError(f"Both Google Drive and GCS download failed. GCS error: {e}")

    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(output_dir)

    zip_path.unlink()

    files = [
        path for path in output_dir.rglob("*")
        if path.is_file()
    ]

    print(f"Dataset extracted to {dataset.path}")
    print(f"Total files: {len(files)}")

In [100]:
@dsl.component(
    base_image=IMAGE_URI
)
def train_model(
    dataset: Input[Dataset],
    train_config: str,
):
    #################### INITIALIZATION ####################
    import argparse
    import json
    import logging
    import math
    import os
    import shutil
    import subprocess
    import sys
    from datetime import datetime, timezone
    from pathlib import Path

    import numpy as np

    from google.cloud import storage

    print("Starting ArcFace training")

    #################### NUMPY COMPATIBILITY ####################
    # Apply NumPy compatibility aliases for MXNet.
    np.long = np.int_
    np.ulong = np.uint

    if not hasattr(np, "bool"):
        np.bool = bool

    #################### LOAD TRAINING CONFIG ####################
    config_values = json.loads(train_config)

    if not isinstance(config_values, dict):
        raise ValueError("train_config must be a JSON object")

    gcs_checkpoint_path = config_values.pop(
        "gcs_checkpoint_path",
        None,
    )

    required_keys = {
        "margin_list",
        "network",
        "resume",
        "embedding_size",
        "sample_rate",
        "fp16",
        "weight_decay",
        "batch_size",
        "lr",
        "verbose",
        "dali",
        "dali_aug",
        "save_all_states",
        "num_classes",
        "num_image",
        "num_epoch",
        "warmup_epoch",
        "val_targets",
        "seed",
        "optimizer",
        "gradient_acc",
        "num_workers",
        "frequent",
        "interclass_filtering_threshold",
    }

    missing_keys = required_keys - config_values.keys()

    if missing_keys:
        raise ValueError(
            f"Missing training config keys: {sorted(missing_keys)}"
        )

    if not isinstance(config_values["resume"], bool):
        raise ValueError("resume must be a boolean")

    if config_values["resume"]:
        if (
            not isinstance(gcs_checkpoint_path, str)
            or not gcs_checkpoint_path.startswith("gs://")
            or "/" not in gcs_checkpoint_path[5:]
        ):
            raise ValueError(
                "resume=True requires a valid "
                "gcs_checkpoint_path: gs://bucket/object"
            )

    if config_values["batch_size"] < 1:
        raise ValueError("batch_size must be greater than zero")

    if config_values["gradient_acc"] < 1:
        raise ValueError("gradient_acc must be greater than zero")

    if config_values["num_epoch"] < 1:
        raise ValueError("num_epoch must be greater than zero")

    output_dir = Path(
        config_values.get("output") or "/tmp/arcface_output"
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    config_values["output"] = str(output_dir)

    #################### CLONE ARCFACE REPOSITORY ####################
    repo_url = "https://github.com/ridwanFatur/arcface-torch-fixed"
    root = Path("/tmp/arcface_torch")

    if root.exists():
        shutil.rmtree(root)

    subprocess.run(
        ["git", "clone", "--depth", "1", repo_url, str(root)],
        check=True,
    )

    sys.path.insert(0, str(root))

    #################### IMPORT ML LIBRARIES ####################
    import mxnet as mx
    import torch
    import torchvision

    from torch import distributed
    from torch.distributed.algorithms.ddp_comm_hooks.default_hooks import (
        fp16_compress_hook,
    )
    from torch.utils.data import DataLoader
    from torch.utils.tensorboard import SummaryWriter

    #################### IMPORT ARCFACE MODULES ####################
    from backbones import get_model
    from dataset import get_dataloader
    from losses import CombinedMarginLoss
    from lr_scheduler import PolynomialLRWarmup
    from partial_fc_v2 import PartialFC_V2
    from utils.utils_callbacks import CallBackLogging, CallBackVerification
    from utils.utils_config import get_config
    from utils.utils_distributed_sampler import setup_seed
    from utils.utils_logging import AverageMeter, init_logging

    #################### ENVIRONMENT DIAGNOSTICS ####################
    print(
        subprocess.run(
            ["nvidia-smi"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout
    )

    print(f"Python: {sys.version}")
    print(f"NumPy: {np.__version__}")
    print(f"PyTorch: {torch.__version__}")
    print(f"Torchvision: {torchvision.__version__}")
    print(f"MXNet: {mx.__version__}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"GPU count: {torch.cuda.device_count()}")

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable. GPU training cannot start.")

    #################### LOCATE DATASET ####################
    dataset_dir = Path(dataset.path)

    rec_files = list(dataset_dir.rglob("train.rec"))
    idx_files = list(dataset_dir.rglob("train.idx"))
    bin_files = list(dataset_dir.rglob("*.bin"))

    print(f"REC files: {len(rec_files)}")
    print(f"IDX files: {len(idx_files)}")
    print(f"BIN files: {len(bin_files)}")

    if not rec_files or not idx_files:
        raise FileNotFoundError(
            f"train.rec or train.idx not found under {dataset_dir}"
        )

    rec_path = rec_files[0]

    idx_path = next(
        (path for path in idx_files if path.parent == rec_path.parent),
        None,
    )

    if idx_path is None:
        raise FileNotFoundError(
            f"Matching train.idx not found for {rec_path}"
        )

    train_root = rec_path.parent

    print(f"Dataset root: {train_root}")
    print(f"REC path: {rec_path}")
    print(f"IDX path: {idx_path}")

    # Always use the dataset provided by the pipeline.
    config_values["rec"] = str(train_root)

    #################### WRITE ARCFACE CONFIG ####################
    config_dir = root / "configs"
    config_dir.mkdir(parents=True, exist_ok=True)

    config_path = config_dir / "faces_webface_vit_s.py"

    config_content = (
        "from easydict import EasyDict as edict\n\n"
        f"config = edict({config_values!r})\n"
    )

    config_path.write_text(config_content, encoding="utf-8")

    print(f"Config path: {config_path}")
    print(config_path.read_text(encoding="utf-8"))

    #################### INITIALIZE DISTRIBUTED TRAINING ####################
    rank = 0
    local_rank = 0
    world_size = 1

    torch.cuda.set_device(local_rank)

    distributed.init_process_group(
        backend="nccl",
        init_method="tcp://127.0.0.1:12584",
        rank=rank,
        world_size=world_size,
    )

    torch.backends.cudnn.benchmark = True

    #################### LOAD ARCFACE CONFIG ####################
    parser = argparse.ArgumentParser(
        description="Distributed ArcFace Training in PyTorch"
    )
    parser.add_argument("config", type=str, help="Python config file")

    args = parser.parse_args([str(config_path)])
    cfg = get_config(args.config)

    setup_seed(
        seed=cfg.seed,
        cuda_deterministic=False,
    )

    os.makedirs(cfg.output, exist_ok=True)
    init_logging(rank, cfg.output)

    summary_writer = (
        SummaryWriter(
            log_dir=os.path.join(cfg.output, "tensorboard")
        )
        if rank == 0
        else None
    )

    wandb_logger = None

    #################### DOWNLOAD CHECKPOINT FROM GCS ####################
    checkpoint_path = os.path.join(
        cfg.output,
        f"checkpoint_gpu_{rank}.pt",
    )

    if cfg.resume:
        bucket_name, blob_name = gcs_checkpoint_path[5:].split("/", 1)

        client = storage.Client()
        bucket = client.bucket(bucket_name)
        blob = bucket.blob(blob_name)

        if not blob.exists():
            raise FileNotFoundError(
                f"GCS checkpoint does not exist: {gcs_checkpoint_path}"
            )

        blob.download_to_filename(checkpoint_path)

        print(f"Checkpoint downloaded: {gcs_checkpoint_path}")
        print(f"Local checkpoint: {checkpoint_path}")

    #################### CREATE DATA LOADER ####################
    train_loader = get_dataloader(
        str(train_root),
        local_rank,
        cfg.batch_size,
        cfg.dali,
        cfg.dali_aug,
        cfg.seed,
        cfg.num_workers,
    )

    #################### INITIALIZE BACKBONE ####################
    backbone = get_model(
        cfg.network,
        dropout=0.0,
        fp16=cfg.fp16,
        num_features=cfg.embedding_size,
    ).cuda()

    backbone = torch.nn.parallel.DistributedDataParallel(
        module=backbone,
        broadcast_buffers=False,
        device_ids=[local_rank],
        bucket_cap_mb=16,
        find_unused_parameters=True,
    )

    backbone.register_comm_hook(None, fp16_compress_hook)
    backbone.train()

    #################### INITIALIZE LOSS ####################
    margin_loss = CombinedMarginLoss(
        64,
        cfg.margin_list[0],
        cfg.margin_list[1],
        cfg.margin_list[2],
        cfg.interclass_filtering_threshold,
    )

    module_partial_fc = PartialFC_V2(
        margin_loss,
        cfg.embedding_size,
        cfg.num_classes,
        cfg.sample_rate,
        False,
    )

    module_partial_fc.train().cuda()

    #################### INITIALIZE OPTIMIZER ####################
    optimizer_params = [
        {"params": backbone.parameters()},
        {"params": module_partial_fc.parameters()},
    ]

    if cfg.optimizer == "sgd":
        opt = torch.optim.SGD(
            params=optimizer_params,
            lr=cfg.lr,
            momentum=cfg.momentum,
            weight_decay=cfg.weight_decay,
        )

    elif cfg.optimizer == "adamw":
        opt = torch.optim.AdamW(
            params=optimizer_params,
            lr=cfg.lr,
            weight_decay=cfg.weight_decay,
        )

    else:
        raise ValueError(
            f"Unsupported optimizer: {cfg.optimizer}"
        )

    #################### INITIALIZE LEARNING RATE SCHEDULER ####################
    cfg.total_batch_size = cfg.batch_size * world_size

    cfg.warmup_step = (
        cfg.num_image // cfg.total_batch_size * cfg.warmup_epoch
    )

    batches_per_epoch = math.ceil(
        cfg.num_image / cfg.total_batch_size
    )

    optimizer_steps_per_epoch = math.ceil(
        batches_per_epoch / cfg.gradient_acc
    )

    cfg.total_step = optimizer_steps_per_epoch * cfg.num_epoch

    lr_scheduler = PolynomialLRWarmup(
        optimizer=opt,
        warmup_iters=cfg.warmup_step,
        total_iters=cfg.total_step,
    )

    #################### INITIALIZE TRAINING STATE ####################
    start_epoch = 0
    global_step = 0
    last_epoch = 0

    #################### RESUME TRAINING ####################
    if cfg.resume:
        checkpoint = torch.load(
            checkpoint_path,
            map_location=f"cuda:{local_rank}",
            weights_only=False,
        )

        required_checkpoint_keys = {
            "epoch",
            "global_step",
            "state_dict_backbone",
            "state_dict_softmax_fc",
            "state_optimizer",
            "state_lr_scheduler",
        }

        missing_checkpoint_keys = (
            required_checkpoint_keys - checkpoint.keys()
        )

        if missing_checkpoint_keys:
            raise ValueError(
                "Invalid checkpoint. Missing keys: "
                f"{sorted(missing_checkpoint_keys)}"
            )

        start_epoch = checkpoint["epoch"]
        global_step = checkpoint["global_step"]

        backbone.module.load_state_dict(
            checkpoint["state_dict_backbone"]
        )

        module_partial_fc.load_state_dict(
            checkpoint["state_dict_softmax_fc"]
        )

        opt.load_state_dict(
            checkpoint["state_optimizer"]
        )

        lr_scheduler.load_state_dict(
            checkpoint["state_lr_scheduler"]
        )

        last_epoch = start_epoch

        del checkpoint

        print(f"Resumed epoch: {start_epoch}")
        print(f"Resumed global step: {global_step}")

    #################### LOG TRAINING CONFIG ####################
    for key, value in cfg.items():
        logging.info("%-25s %s", key, value)

    if start_epoch >= cfg.num_epoch:
        print(
            f"Training already reached the target epoch: "
            f"{start_epoch}/{cfg.num_epoch}"
        )

    #################### INITIALIZE CALLBACKS ####################
    callback_verification = CallBackVerification(
        val_targets=cfg.val_targets,
        rec_prefix=cfg.rec,
        summary_writer=summary_writer,
        wandb_logger=wandb_logger,
    )

    callback_logging = CallBackLogging(
        frequent=cfg.frequent,
        total_step=cfg.total_step,
        batch_size=cfg.batch_size,
        start_step=global_step,
        writer=summary_writer,
    )

    #################### INITIALIZE METRICS AND AMP ####################
    loss_am = AverageMeter()

    amp = torch.cuda.amp.GradScaler(
        enabled=cfg.fp16,
        growth_interval=100,
    )

    opt.zero_grad(set_to_none=True)

    #################### TRAIN MODEL ####################
    try:
        for epoch in range(start_epoch, cfg.num_epoch):
            last_epoch = epoch + 1

            if hasattr(train_loader.sampler, "set_epoch"):
                train_loader.sampler.set_epoch(epoch)

            num_batches = len(train_loader)

            for batch_idx, (img, local_labels) in enumerate(train_loader):
                global_step += 1

                is_accumulation_step = (
                    global_step % cfg.gradient_acc == 0
                )

                is_last_batch = batch_idx + 1 == num_batches

                should_update = (
                    is_accumulation_step or is_last_batch
                )

                #################### FORWARD PASS ####################
                local_embeddings = backbone(img)

                loss = module_partial_fc(
                    local_embeddings,
                    local_labels,
                )

                #################### BACKWARD PASS ####################
                scaled_loss = loss / cfg.gradient_acc

                if cfg.fp16:
                    amp.scale(scaled_loss).backward()
                else:
                    scaled_loss.backward()

                #################### OPTIMIZER UPDATE ####################
                if should_update:
                    if cfg.fp16:
                        amp.unscale_(opt)

                    torch.nn.utils.clip_grad_norm_(
                        backbone.parameters(),
                        max_norm=5.0,
                    )

                    torch.nn.utils.clip_grad_norm_(
                        module_partial_fc.parameters(),
                        max_norm=5.0,
                    )

                    if cfg.fp16:
                        amp.step(opt)
                        amp.update()
                    else:
                        opt.step()

                    opt.zero_grad(set_to_none=True)

                    lr_scheduler.step()

                #################### UPDATE METRICS ####################
                with torch.no_grad():
                    loss_am.update(loss.item(), 1)

                    if wandb_logger:
                        wandb_logger.log(
                            {
                                "Loss/Step Loss": loss.item(),
                                "Loss/Train Loss": loss_am.avg,
                                "Process/Step": global_step,
                                "Process/Epoch": epoch,
                            }
                        )

                    callback_logging(
                        global_step,
                        loss_am,
                        epoch,
                        cfg.fp16,
                        lr_scheduler.get_last_lr()[0],
                        amp,
                    )

                    if (
                        cfg.verbose > 0
                        and global_step % cfg.verbose == 0
                    ):
                        callback_verification(
                            global_step,
                            backbone,
                        )

            print(
                f"Epoch completed: {last_epoch}/{cfg.num_epoch}"
            )

    #################### CLEANUP ON TRAINING ERROR ####################
    except Exception:
        if summary_writer is not None:
            summary_writer.close()

        raise

    #################### SAVE CHECKPOINT ####################
    if cfg.save_all_states:
        checkpoint_path = os.path.join(
            cfg.output,
            f"checkpoint_gpu_{rank}.pt",
        )

        checkpoint = {
            "epoch": last_epoch,
            "global_step": global_step,
            "state_dict_backbone": backbone.module.state_dict(),
            "state_dict_softmax_fc": module_partial_fc.state_dict(),
            "state_optimizer": opt.state_dict(),
            "state_lr_scheduler": lr_scheduler.state_dict(),
        }

        torch.save(checkpoint, checkpoint_path)

        print(f"Checkpoint saved: {checkpoint_path}")

        #################### UPLOAD CHECKPOINT TO GCS ####################
        timestamp = datetime.now(timezone.utc).strftime(
            "%Y%m%dT%H%M%S%fZ"
        )

        blob_name = (
            f"artifacts/{timestamp}/"
            f"checkpoint_gpu_{rank}.pt"
        )

        bucket_name = "rf-pytorch-bucket"

        client = storage.Client()
        bucket = client.bucket(bucket_name)
        blob = bucket.blob(blob_name)

        blob.upload_from_filename(checkpoint_path)

        print(
            f"Checkpoint uploaded: "
            f"gs://{bucket_name}/{blob_name}"
        )

    #################### CLEANUP ####################
    if summary_writer is not None:
        summary_writer.close()

    distributed.destroy_process_group()

    print("ArcFace training completed")

In [106]:
@dsl.pipeline(name="arcface-training-pipeline")
def pipeline(train_config: str):
    #################### DOWNLOAD DATASET ####################
    data_task = download_data()

    #################### TRAIN MODEL ####################
    train_task = train_model(
        dataset=data_task.outputs["dataset"],
        train_config=train_config,
    )

    #################### CONFIGURE COMPUTE RESOURCES ####################
    train_task.set_cpu_limit("4")
    train_task.set_memory_limit("16Gi")
    train_task.set_accelerator_type("NVIDIA_TESLA_T4")
    train_task.set_accelerator_limit(1)

In [107]:
compiler.Compiler().compile(
    pipeline_func=pipeline,
    package_path="pipeline.yaml",
)

In [108]:
import json

train_config = json.dumps({
    "margin_list": [1.0, 0.0, 0.4],
    "network": "vit_s",
    "resume": True,
    "gcs_checkpoint_path": (
        "gs://rf-pytorch-bucket/artifacts/20261020T093555543367Z/checkpoint_gpu_0.pt"
    ),
    "output": "/tmp/arcface_output",
    "embedding_size": 512,
    "sample_rate": 1.0,
    "fp16": True,
    "momentum": 0.9,
    "weight_decay": 1e-4,
    "batch_size": 128,
    "lr": 0.1,
    "verbose": 2000,
    "dali": False,
    "save_all_states": True,
    "num_classes": 10572,
    "num_image": 490623,
    "num_epoch": 3,
    "warmup_epoch": 0,
    "val_targets": ["lfw"],

    "seed": 42,
    "optimizer": "sgd",
    "gradient_acc": 1,
    "dali_aug": False,
    "num_workers": 4,
    "frequent": 10,
    "interclass_filtering_threshold": 0,
})

In [109]:
job = aiplatform.PipelineJob(
    display_name="pipeline",
    template_path="pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
         "train_config": train_config
    },
    location="us-central1",
)

In [110]:
SERVICE_ACCOUNT = os.environ["SERVICE_ACCOUNT"]
job.submit(service_account = SERVICE_ACCOUNT)

Creating PipelineJob
PipelineJob created. Resource name: projects/344969539300/locations/us-central1/pipelineJobs/arcface-training-pipeline-20261020185142
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/344969539300/locations/us-central1/pipelineJobs/arcface-training-pipeline-20261020185142')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/arcface-training-pipeline-20261020185142?project=344969539300


### Logs

In [77]:
from google.cloud import logging
client = logging.Client()

In [95]:
filter_ = '''
'''

In [84]:
logs = list(client.list_entries(filter_=filter_))

In [85]:
len(logs)

76

In [94]:
# for i in range(0, len(logs)):
#     print(logs[i].payload)